## AWS Setup

In [ ]:
# ── AWS credentials + S3 constants ───────────────────────────────────────────
# Store credentials as Colab secrets (Secrets panel in the left sidebar).
# Add two secrets named  AWS_ACCESS_KEY_ID  and  AWS_SECRET_ACCESS_KEY,
# then enable notebook access for each.  They will be read below.

!pip install -q datasets boto3 pandas huggingface_hub

from google.colab import userdata
import os

os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = "ap-south-1"

S3_BUCKET = "nilesh-rl"
S3_PREFIX = "maithili_rl_project"

import boto3
s3 = boto3.client("s3")
print("AWS credentials loaded ✓")

## Data And AWS

In [2]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from datasets import load_dataset, DatasetDict

# Load Flores-200 dataset for English (eng_Latn) and Maithili (mai_Deva) from Hugging Face
# The 'flores_plus' dataset on Hugging Face uses 'eng_Latn' for English and 'mai_Deva' for Maithili.
eng_dataset = load_dataset("openlanguagedata/flores_plus", "eng_Latn")
mai_dataset = load_dataset("openlanguagedata/flores_plus", "mai_Deva")

# The dataset already has a 'sentence' column, so no renaming is needed.
# We can directly access the 'dev' and 'devtest' splits.

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

eng_Latn.jsonl: 0.00B [00:00, ?B/s]

eng_Latn.jsonl: 0.00B [00:00, ?B/s]

...

mai_Deva.jsonl: 0.00B [00:00, ?B/s]

mai_Deva.jsonl: 0.00B [00:00, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

NameError: name 'eng_dataset_dict' is not defined

In [4]:
print("English DatasetDict splits:", eng_dataset.keys())
print("Maithili DatasetDict splits:", mai_dataset.keys())

English DatasetDict splits: dict_keys(['dev', 'devtest'])
Maithili DatasetDict splits: dict_keys(['dev', 'devtest'])


In [5]:
# ── Cell 4: Create Parallel DataFrames ──────────────────────────────

import pandas as pd

dev_df = pd.DataFrame({
"english":  eng_dataset["dev"]["text"],
"maithili": mai_dataset["dev"]["text"],
})

devtest_df = pd.DataFrame({
"english":  eng_dataset["devtest"]["text"],
"maithili": mai_dataset["devtest"]["text"],
})

# Combine

full_parallel = pd.concat(
[dev_df, devtest_df],
ignore_index=True
)

# Shuffle

full_parallel = full_parallel.sample(
frac=1,
random_state=42
).reset_index(drop=True)

print("Total parallel samples:", len(full_parallel))

Total parallel samples: 2009


In [6]:
# ── Create Splits ───────────────────────────────────────────

# Validation/Test

valid_df = full_parallel.iloc[:500]
test_df  = full_parallel.iloc[500:1000]

# RL Monolingual English Train Data (~10k)

train_en = pd.concat([
full_parallel["english"],
full_parallel["english"],
full_parallel["english"],
full_parallel["english"],
full_parallel["english"],
], ignore_index=True)

train_en = train_en.sample(
n=10000,
replace=True,
random_state=42
).reset_index(drop=True)

print("Train EN:", len(train_en))
print("Valid:", len(valid_df))
print("Test:", len(test_df))

Train EN: 10000
Valid: 500
Test: 500


In [7]:
# ──Save Files ──────────────────────────────────────────────

# RL training file

with open("train.en.txt", "w", encoding="utf-8") as f:
    for line in train_en:
        f.write(line.strip() + "\n")

# Validation/Test parallel files

valid_df.to_csv("valid_parallel.csv", index=False)
test_df.to_csv("test_parallel.csv", index=False)

print("Saved locally ✓")

Saved locally ✓


In [8]:
# ── Upload to S3 ────────────────────────────────────────────

import boto3

s3 = boto3.client("s3")

files_to_upload = [
"train.en.txt",
"valid_parallel.csv",
"test_parallel.csv",
]

for fname in files_to_upload:
    key = f"{S3_PREFIX}/{fname}"
    s3.upload_file(fname, S3_BUCKET, key)
    print(f"Uploaded → s3://{S3_BUCKET}/{key}")

print("All uploads complete ✓")

Uploaded → s3://nilesh-rl/maithili_rl_project/train.en.txt
Uploaded → s3://nilesh-rl/maithili_rl_project/valid_parallel.csv
Uploaded → s3://nilesh-rl/maithili_rl_project/test_parallel.csv
All uploads complete ✓


## Training Experiments

**Run order (each Colab session):**

1. **Cell: AWS Setup** (cell-1 — credentials + S3 constants)
2. **Cell: Setup** (cell-11 — clone repo + install deps)
3. **Cell: Data** (cell-15 — fetch from S3, auto-fixes columns on first run)
4. **One experiment cell only** — choose the one matching this session:
   - `Experiment 1` → Baseline eval (Session 1)
   - `Experiment 2` → RL Baseline (Session 2)
   - `Experiment 3` → Modified RL (Session 3)

> The Data cell is at the bottom of this section. **Jump to it and run it before starting any experiment.**

In [ ]:
# ── SETUP: Clone repo + install dependencies ─────────────────────
import os

repo_path = "/content/MT-via-Round-Trip-RL"

if not os.path.exists(repo_path):
    !git clone https://github.com/thenileshmishra/MT-via-Round-Trip-RL.git /content/MT-via-Round-Trip-RL
else:
    print("Repo already present, pulling latest...")
    !git -C {repo_path} pull

# Install dependencies (torchdata pinned to avoid datapipes removal)
!pip install -q -r {repo_path}/requirements.txt
!pip install -q torchdata==0.7.1

# Quick sanity check
from torchdata.datapipes.map import MapDataPipe
print("Setup complete ✓")

In [ ]:
# ── EXPERIMENT 1: Baseline  (vanilla NLLB-600M, eval only — no training) ─────
# Run this cell alone on Session 1.
# Prereqs: cell-1 (AWS creds) + cell-11 (Setup) + cell-15 (Data) must be run first.

import subprocess, os, time

REPO = "/content/MT-via-Round-Trip-RL"
EXP_NAME = "baseline_eval_maithili"

# ── Pre-flight checks ────────────────────────────────────────────────────────
assert os.path.exists(f"{REPO}/configs/task/nllb_maithili.yaml"), \
    f"Config not found — did `git pull` in cell-11 succeed?"
for f in ["mai_train.csv", "mai_valid.csv", "mai_test.csv"]:
    assert os.path.exists(f"/content/{f}"), f"/content/{f} missing — run cell-15 first"

os.chdir(REPO)
print(f"Running experiment: {EXP_NAME}")
print("Output will stream below — watch for the startup banner to confirm model/data...\n")

t0 = time.time()
result = subprocess.run([
    "python", "main.py",
    "task=nllb_maithili",
    "task.model.name=facebook/nllb-200-distilled-600M",
    "task.experiment.mode=baseline_eval",
    f"task.experiment.name={EXP_NAME}",
    f"task.experiment.s3_output_uri=s3://{S3_BUCKET}/{S3_PREFIX}",
    "task.data.train_file=/content/mai_train.csv",
    "task.data.valid_file=/content/mai_valid.csv",
    "task.data.test_file=/content/mai_test.csv",
    "task.training.use_wandb=false",
])
elapsed = time.time() - t0
print(f"\nElapsed: {elapsed/60:.1f} min  |  Exit code: {result.returncode}")

if result.returncode != 0:
    raise RuntimeError(f"Training process FAILED (exit code {result.returncode}). "
                       "Scroll up to see the error from Python/Hydra.")

# ── Verify output artifact ───────────────────────────────────────────────────
import json, glob
results_json = glob.glob(f"{REPO}/results/{EXP_NAME}*.json")
if results_json:
    with open(results_json[0]) as fh:
        summary = json.load(fh)
    print(f"\nResults saved: {results_json[0]}")
    if summary.get("final_eval"):
        for k, v in summary["final_eval"].items():
            print(f"  eval/{k}: {v:.4f}")
    if summary.get("final_test"):
        for k, v in summary["final_test"].items():
            print(f"  test/{k}: {v:.4f}")
else:
    print(f"\n[WARN] No results JSON found under {REPO}/results/ — check for errors above.")

In [ ]:
# ── EXPERIMENT 2: RL Baseline  (R = chrF++ + BLEU) ───────────────────────────
# Run this cell alone on Session 2.
# Prereqs: cell-1 (AWS creds) + cell-11 (Setup) + cell-15 (Data) must be run first.

import subprocess, os, time

REPO = "/content/MT-via-Round-Trip-RL"
EXP_NAME = "rl_baseline_maithili"

# ── Pre-flight checks ────────────────────────────────────────────────────────
assert os.path.exists(f"{REPO}/configs/task/nllb_maithili.yaml"), \
    f"Config not found — did `git pull` in cell-11 succeed?"
for f in ["mai_train.csv", "mai_valid.csv", "mai_test.csv"]:
    assert os.path.exists(f"/content/{f}"), f"/content/{f} missing — run cell-15 first"

os.chdir(REPO)
print(f"Running experiment: {EXP_NAME}")
print("Output will stream below — watch for the startup banner to confirm model/data/step count...\n")

t0 = time.time()
result = subprocess.run([
    "python", "main.py",
    "task=nllb_maithili",
    "task.model.name=facebook/nllb-200-distilled-600M",
    "task.experiment.mode=rl_baseline",
    f"task.experiment.name={EXP_NAME}",
    f"task.experiment.s3_output_uri=s3://{S3_BUCKET}/{S3_PREFIX}",
    "task.experiment.upload_model_to_s3=true",
    "task.data.train_file=/content/mai_train.csv",
    "task.data.valid_file=/content/mai_valid.csv",
    "task.data.test_file=/content/mai_test.csv",
    "task.training.use_wandb=false",
])
elapsed = time.time() - t0
print(f"\nElapsed: {elapsed/60:.1f} min  |  Exit code: {result.returncode}")

if result.returncode != 0:
    raise RuntimeError(f"Training process FAILED (exit code {result.returncode}). "
                       "Scroll up to see the error from Python/Hydra.")

# ── Verify output artifact ───────────────────────────────────────────────────
import json, glob
results_json = glob.glob(f"{REPO}/results/{EXP_NAME}*.json")
if results_json:
    with open(results_json[0]) as fh:
        summary = json.load(fh)
    print(f"\nResults saved: {results_json[0]}")
    print(f"Training steps logged: {len(summary.get('history', []))}")
    if summary.get("final_eval"):
        for k, v in summary["final_eval"].items():
            print(f"  eval/{k}: {v:.4f}")
    if summary.get("final_test"):
        for k, v in summary["final_test"].items():
            print(f"  test/{k}: {v:.4f}")
else:
    print(f"\n[WARN] No results JSON found under {REPO}/results/ — check for errors above.")

In [ ]:
# ── EXPERIMENT 3: Modified RL  (R = 0.7·chrF++ + 0.2·BLEU + 0.1·LMScore) ────
# Run this cell alone on Session 3.
# Prereqs: cell-1 (AWS creds) + cell-11 (Setup) + cell-15 (Data) must be run first.

import subprocess, os, time

REPO = "/content/MT-via-Round-Trip-RL"
EXP_NAME = "modified_rl_maithili"

# ── Pre-flight checks ────────────────────────────────────────────────────────
assert os.path.exists(f"{REPO}/configs/task/nllb_maithili.yaml"), \
    f"Config not found — did `git pull` in cell-11 succeed?"
for f in ["mai_train.csv", "mai_valid.csv", "mai_test.csv"]:
    assert os.path.exists(f"/content/{f}"), f"/content/{f} missing — run cell-15 first"

os.chdir(REPO)
print(f"Running experiment: {EXP_NAME}")
print("Output will stream below — watch for the startup banner to confirm model/data/step count...\n")

t0 = time.time()
result = subprocess.run([
    "python", "main.py",
    "task=nllb_maithili",
    "task.model.name=facebook/nllb-200-distilled-600M",
    "task.experiment.mode=modified_rl",
    f"task.experiment.name={EXP_NAME}",
    f"task.experiment.s3_output_uri=s3://{S3_BUCKET}/{S3_PREFIX}",
    "task.experiment.upload_model_to_s3=true",
    "task.data.train_file=/content/mai_train.csv",
    "task.data.valid_file=/content/mai_valid.csv",
    "task.data.test_file=/content/mai_test.csv",
    "task.training.use_wandb=false",
])
elapsed = time.time() - t0
print(f"\nElapsed: {elapsed/60:.1f} min  |  Exit code: {result.returncode}")

if result.returncode != 0:
    raise RuntimeError(f"Training process FAILED (exit code {result.returncode}). "
                       "Scroll up to see the error from Python/Hydra.")

# ── Verify output artifact ───────────────────────────────────────────────────
import json, glob
results_json = glob.glob(f"{REPO}/results/{EXP_NAME}*.json")
if results_json:
    with open(results_json[0]) as fh:
        summary = json.load(fh)
    print(f"\nResults saved: {results_json[0]}")
    print(f"Training steps logged: {len(summary.get('history', []))}")
    if summary.get("final_eval"):
        for k, v in summary["final_eval"].items():
            print(f"  eval/{k}: {v:.4f}")
    if summary.get("final_test"):
        for k, v in summary["final_test"].items():
            print(f"  test/{k}: {v:.4f}")
else:
    print(f"\n[WARN] No results JSON found under {REPO}/results/ — check for errors above.")

In [ ]:
# ── DATA: Fetch from S3  (run every session, before any experiment cell) ──────
# Handles three situations automatically:
#   1. Correct files already on S3 → just download
#   2. Files exist but wrong column names → rebuild from FLORES, re-upload
#   3. Files missing entirely (first time, only old train.en.txt exists) → rebuild

import boto3
import pandas as pd
from botocore.exceptions import ClientError

s3 = boto3.client("s3")
REQUIRED = ["mai_train.csv", "mai_valid.csv", "mai_test.csv"]

def _exists_on_s3(fname):
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=f"{S3_PREFIX}/{fname}")
        return True
    except ClientError:
        return False

def _download():
    for fname in REQUIRED:
        s3.download_file(S3_BUCKET, f"{S3_PREFIX}/{fname}", f"/content/{fname}")
        print(f"  Downloaded → /content/{fname}")

def _columns_ok():
    df = pd.read_csv("/content/mai_train.csv", nrows=1)
    return "sentence_eng_Latn" in df.columns and "sentence_mai_Deva" in df.columns

def _rebuild_and_upload():
    from datasets import load_dataset
    print("  Downloading FLORES-200 from HuggingFace ...")
    eng = load_dataset("openlanguagedata/flores_plus", "eng_Latn", trust_remote_code=True)
    mai = load_dataset("openlanguagedata/flores_plus", "mai_Deva", trust_remote_code=True)

    full = pd.DataFrame({
        "sentence_eng_Latn": [r["text"] for r in eng["dev"]] + [r["text"] for r in eng["devtest"]],
        "sentence_mai_Deva": [r["text"] for r in mai["dev"]] + [r["text"] for r in mai["devtest"]],
    }).sample(frac=1, random_state=42).reset_index(drop=True)

    splits = {
        "mai_train.csv": full.iloc[1000:].reset_index(drop=True),  # 1009 rows
        "mai_valid.csv": full.iloc[:500].copy(),                    # 500 rows
        "mai_test.csv":  full.iloc[500:1000].copy(),                # 500 rows
    }
    for fname, df in splits.items():
        df.to_csv(f"/content/{fname}", index=False)
        s3.upload_file(f"/content/{fname}", S3_BUCKET, f"{S3_PREFIX}/{fname}")
        print(f"  Rebuilt + uploaded → s3://{S3_BUCKET}/{S3_PREFIX}/{fname}")

# ── Main logic ────────────────────────────────────────────────────────────────
all_present = all(_exists_on_s3(f) for f in REQUIRED)

if not all_present:
    print("S3 files missing (old format detected) — rebuilding from FLORES ...")
    _rebuild_and_upload()
else:
    print("Downloading existing files from S3 ...")
    _download()
    if not _columns_ok():
        print("Wrong column names detected — rebuilding from FLORES ...")
        _rebuild_and_upload()

df = pd.read_csv("/content/mai_train.csv")
print(f"\nTrain: {len(df)} rows | Columns: {df.columns.tolist()}")
assert _columns_ok(), "Fix failed — check AWS credentials and S3 bucket/prefix."
print("Data ready ✓")